In [1]:
## import libraries
import pandas as pd
import numpy as np

In [2]:
## load dataset
df = pd.read_csv("surge_pricing_data.csv")
df.head()

,Hour,Day_Type,Weather,Location,Local_Event,Demand_Score,Price_Multiplier
0,20,Weekday,Sunny,Madhapur,0,5,1.14
1,3,Weekday,Windy,Hitech City,0,2,0.94
2,16,Weekday,Rainy,Dilsukhnagar,1,7,1.58
3,8,Weekday,Rainy,Kukatpally,1,10,1.91
4,10,Weekday,Sunny,Kukatpally,0,4,1.35


#### Data Cleaning

In [3]:
## Checking for null values
df.isnull().sum()

Hour                0
Day_Type            0
Weather             0
Location            0
Local_Event         0
Demand_Score        0
Price_Multiplier    0
dtype: int64

##### There are no null values

#### Data Engineering

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4694 entries, 0 to 4693
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Hour              4694 non-null   int64  
 1   Day_Type          4694 non-null   str    
 2   Weather           4694 non-null   str    
 3   Location          4694 non-null   str    
 4   Local_Event       4694 non-null   int64  
 5   Demand_Score      4694 non-null   int64  
 6   Price_Multiplier  4694 non-null   float64
dtypes: float64(1), int64(3), str(3)
memory usage: 362.6 KB


In [5]:
# data encoding
from sklearn.preprocessing import LabelEncoder
le_day = LabelEncoder()
le_weather = LabelEncoder()
le_location = LabelEncoder()

df['Day_Type'] = le_day.fit_transform(df['Day_Type'])
df['Weather'] = le_weather.fit_transform(df['Weather'])
df['Location'] = le_location.fit_transform(df['Location'])

In [6]:
import pickle

In [7]:
# Save encoders for use in Streamlit app later
pickle.dump(le_day,      open("le_day.pkl",      "wb"))
pickle.dump(le_weather,  open("le_weather.pkl",  "wb"))
pickle.dump(le_location, open("le_location.pkl", "wb"))
print("\nEncoders saved!")


Encoders saved!


#### train test split

In [8]:
df.head()

,Hour,Day_Type,Weather,Location,Local_Event,Demand_Score,Price_Multiplier
0,20,0,2,4,0,5,1.14
1,3,0,3,2,0,2,0.94
2,16,0,1,1,1,7,1.58
3,8,0,1,3,1,10,1.91
4,10,0,2,3,0,4,1.35


In [9]:
## train-test split
from sklearn.model_selection import train_test_split
X = df.drop("Price_Multiplier", axis=1)
y = df["Price_Multiplier"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining rows : {len(X_train)}")
print(f"Testing rows  : {len(X_test)}")


Training rows : 3755
Testing rows  : 939


In [10]:
X_train

,Hour,Day_Type,Weather,Location,Local_Event,Demand_Score
618,3,0,3,1,1,5
3837,13,0,1,1,1,10
2450,18,1,1,0,1,12
3733,21,1,2,2,1,12
4151,22,1,1,4,0,5
...,...,...,...,...,...,...
4426,19,1,3,1,1,8
466,3,1,1,0,1,11
3092,22,0,0,0,1,9
3772,23,0,2,3,0,3


In [11]:
y_train

618     1.36
3837    1.96
2450    1.89
3733    1.96
4151    1.36
        ... 
4426    1.61
466     2.26
3092    2.09
3772    1.17
860     1.04
Name: Price_Multiplier, Length: 3755, dtype: float64

In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [13]:
# Compare Multiple Models First (Without Hyperparameter Tuning)
print("\n--- Comparing Base Models ---")

base_models = {
    "Linear Regression"    : LinearRegression(),
    "Ridge Regression"     : Ridge(),
    "Decision Tree"        : DecisionTreeRegressor(random_state=42),
    "Random Forest"        : RandomForestRegressor(random_state=42),
    "Gradient Boosting"    : GradientBoostingRegressor(random_state=42)
}

results = {}

for name, model in base_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    results[name] = {"MAE": round(mae, 4), "R2": round(r2, 4)}
    print(f"{name:25s} → MAE: {mae:.4f} | R2: {r2:.4f}")


--- Comparing Base Models ---
Linear Regression         → MAE: 0.1393 | R2: 0.7761
Ridge Regression          → MAE: 0.1393 | R2: 0.7761
Decision Tree             → MAE: 0.1364 | R2: 0.7760
Random Forest             → MAE: 0.1097 | R2: 0.8599
Gradient Boosting         → MAE: 0.1017 | R2: 0.8903


In [14]:
## 6. Pick Best Model Automatically
best_model_name = max(results, key=lambda x: results[x]["R2"])
print(f"\n✅ Best Base Model: {best_model_name}")
print(f"   R2: {results[best_model_name]['R2']} | MAE: {results[best_model_name]['MAE']}")



✅ Best Base Model: Gradient Boosting
   R2: 0.8903 | MAE: 0.1017


In [16]:
param_grids = {
    "Random Forest": {
        "n_estimators"      : [100, 200, 300, 500],
        "max_depth"         : [None, 5, 10, 20, 30],
        "min_samples_split" : [2, 5, 10],
        "min_samples_leaf"  : [1, 2, 4],
        "max_features"      : ["sqrt", "log2"]
    },
    "Gradient Boosting": {
        "n_estimators"  : [100, 200, 300],
        "learning_rate" : [0.01, 0.05, 0.1, 0.2],
        "max_depth"     : [3, 5, 7, 10],
        "subsample"     : [0.7, 0.8, 0.9, 1.0]
    },
    "Decision Tree": {
        "max_depth"         : [None, 5, 10, 20, 30],
        "min_samples_split" : [2, 5, 10],
        "min_samples_leaf"  : [1, 2, 4]
    },
    "Ridge Regression": {
        "alpha": [0.01, 0.1, 1, 10, 100]
    },
    "Linear Regression": {}
}


In [17]:
from sklearn.model_selection import RandomizedSearchCV

In [23]:

# Get the model object for best model
best_base_model  = base_models[best_model_name]
best_param_grid  = param_grids[best_model_name]

if best_param_grid:
    search = RandomizedSearchCV(
        estimator  = best_base_model,
        param_distributions = best_param_grid,
        n_iter     = 100,          # Try 100 random combinations
        cv         = 5,           # 5 fold cross validation
        scoring    = "r2",
        random_state = 42,
        n_jobs     = -1           # Use all CPU cores
    )
    search.fit(X_train, y_train)

    best_model  = search.best_estimator_
    best_params = search.best_params_

    print(f"\nBest Parameters Found:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")
else:
    best_model = best_base_model
    best_model.fit(X_train, y_train)
    best_params = {}
    print("Linear Regression has no hyperparameters to tune.")


Best Parameters Found:
  subsample: 1.0
  n_estimators: 100
  max_depth: 3
  learning_rate: 0.05


In [24]:
# 8. Final Model Performance
y_pred_final = best_model.predict(X_test)

final_mae = mean_absolute_error(y_test, y_pred_final)
final_r2  = r2_score(y_test, y_pred_final)

print(f"\n--- Final Tuned Model Performance ---")
print(f"Model : {best_model_name}")
print(f"MAE   : {final_mae:.4f}  (lower is better)")
print(f"R2    : {final_r2:.4f}  (closer to 1.0 is better)")


--- Final Tuned Model Performance ---
Model : Gradient Boosting
MAE   : 0.1016  (lower is better)
R2    : 0.8907  (closer to 1.0 is better)


In [25]:
# 9. Cross Validation Score
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_model, X, y, cv=5, scoring="r2")
print(f"\nCross Validation R2 Scores : {cv_scores.round(4)}")
print(f"Average CV R2 Score        : {cv_scores.mean():.4f}")


Cross Validation R2 Scores : [0.8852 0.8826 0.8832 0.8819 0.8763]
Average CV R2 Score        : 0.8818


In [26]:
#  10. Save the Best Model
pickle.dump(best_model, open("model.pkl", "wb"))
print(f"\n Best model saved as model.pkl!")


 Best model saved as model.pkl!
